In [1]:
import numpy as np
from PIL import Image 
import os

In [2]:
def generate_fork_grating(
    width: int = 1920,
    height: int = 1080,
    topological_charge: int = 3,
    grating_period: float = 24.0,  # Período da grade em pixels (Lambda)
    tilt_angle_deg: float = 0.0,   # Direção da deflexão (0° = horizontal)
    center_x: float = None,
    center_y: float = None,
) -> np.ndarray:
    """
    Gera uma máscara fork grating discretizada em escala de cinza de 8 bits (0 a 255).
    """
    if center_x is None:
        center_x = width / 2.0
    if center_y is None:
        center_y = height / 2.0

    # 1. Malha de coordenadas espaciais
    x = np.arange(width, dtype=np.float64) - center_x
    y = np.arange(height, dtype=np.float64) - center_y
    X, Y = np.meshgrid(x, y)

    # Inversão do eixo Y se necessário para coincidir com a convenção de coordenadas de imagem (origem no topo)
    # Aqui Y cresce para baixo, logo -Y faz o eixo vertical físico apontar para cima
    Y_phys = -Y

    # 2. Vórtice azimutal: l * phi
    phi = np.arctan2(Y_phys, X)
    vortex_phase = topological_charge * phi

    # 3. Grade portadora (Blazed Grating)
    theta = np.deg2rad(tilt_angle_deg)
    # Projeção ao longo da direção de deflexão
    carrier_phase = (2.0 * np.pi / grating_period) * (X * np.cos(theta) + Y_phys * np.sin(theta))

    # 4. Fase total envelopada em [0, 2*pi)
    total_phase = (vortex_phase + carrier_phase) % (2.0 * np.pi)

    # 5. Mapeamento linear para 8 bits [0, 255]
    phase_pattern_uint8 = np.round((total_phase / (2.0 * np.pi)) * 255.0).astype(np.uint8)

    return phase_pattern_uint8


In [ ]:
import matplotlib.pyplot as plt

# Faz o download do arquivo do servidor remoto para o seu computador local

if __name__ == "__main__":
    l = 3
    period_px = 30
    w, h = 1920, 1080

    mask = generate_fork_grating(
        width=w, 
        height=h, 
        topological_charge=l, 
        grating_period=period_px,
        tilt_angle_deg=0.0
    )

    # Cria a imagem deixando o Pillow inferir o modo pelo np.uint8
    img = Image.fromarray(mask)
    
    # Salva com caminho absoluto para garantir que você saiba onde foi parar
    nome_arquivo = f"fork_grating_l{l}.png"
    caminho_absoluto = os.path.abspath(nome_arquivo)
    #plt.figure(figsize=(10,6))
    #plt.imshow(mask, cmap='gray', vmin=0, vmax=255)
    #plt.imsave(f"fork_grating_l{l}.png", mask, cmap='gray')
    
    print(f"Salvo exatamente em: {caminho_absoluto}")
    img.save(caminho_absoluto)
    #print(f"Máscara gerada com sucesso!")
    #print(f"Salvo exatamente em: {caminho_absoluto}")

In [5]:
print(caminho_absoluto)

/home/pedro/Dev/Playground/util/jupiter/phase_masks/l3/mask1_fork_grating_l3/fork_grating_l3.png
